In [ ]:
import cv2
import os

frames_folder = "frames"
output_folder = "tracked_frames"
os.makedirs(output_folder, exist_ok=True)

# Get only image frames
frames = sorted([f for f in os.listdir(frames_folder) if f.endswith(".jpg") or f.endswith(".png")])

# Read first frame
first_frame_path = os.path.join(frames_folder, frames[0])
frame = cv2.imread(first_frame_path)

if frame is None:
    print("Error loading first frame")
    exit()

# Select player
bbox = cv2.selectROI("Select Player", frame, False)
cv2.destroyWindow("Select Player")

# Create tracker
tracker = cv2.legacy.TrackerCSRT_create()

# Initialize
tracker.init(frame, bbox)

frame_count = 0

for frame_name in frames:

    frame_path = os.path.join(frames_folder, frame_name)
    frame = cv2.imread(frame_path)

    if frame is None:
        continue

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = [int(v) for v in bbox]

        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)

        cv2.putText(frame, "Player Tracking",
                    (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0,255,0), 2)
    else:
        cv2.putText(frame, "Tracking Failed",
                    (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0,0,255), 2)

    save_path = os.path.join(output_folder, f"tracked_{frame_count}.jpg")
    cv2.imwrite(save_path, frame)

    cv2.imshow("Tracking", frame)

    if cv2.waitKey(30) & 0xFF == 27:
        break

    frame_count += 1

cv2.destroyAllWindows()